# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUKUL-TIWARI/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
%pip -q install duckdb huggingface_hub pandas numpy

In [7]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [8]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully!")

Connected successfully!


In [9]:
con.sql(f"""
SELECT *
FROM {TABLES["fact_daily"]}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [10]:
con.sql(f"""
SELECT COUNT(*)
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
""").df()

,count_star()
0,9841378


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule

A content page should be reviewed if it is visible in Google Search, has not been receiving engagement through GA4, and still has measurable search impressions. These pages represent opportunities where content is discoverable but may no longer be performing well.

### Reason code

- `visible_low_engagement`

### Action label

- `REVIEW_CONTENT`

This baseline is intentionally simple and transparent. It serves as a human-readable rule that the Week 5 machine learning model should improve upon.

In [11]:
query = f"""
SELECT
    COUNT(*) AS total_pages,
    COUNT(*) FILTER (WHERE gsc_impressions >= 100) AS visible_pages,
    COUNT(*) FILTER (
        WHERE gsc_impressions >= 100
          AND ga4_engaged_sessions = 0
    ) AS visible_low_engagement
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
"""

con.sql(query).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_pages,visible_pages,visible_low_engagement
0,9841378,638608,361304


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Baseline scoring rule

The baseline assigns higher scores to pages that are visible in Google Search but receive little engagement. Pages with more search impressions are prioritized because improving them may have a larger impact.

Reason code:
- `visible_low_engagement`

Action label:
- `REVIEW_CONTENT`

In [14]:
import os
import numpy as np

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_avg_position,
    COALESCE(ga4_engaged_sessions,0) AS ga4_engaged_sessions
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
  AND gsc_impressions >= 100
"""

df = con.sql(query).df()

# Baseline score
df["score"] = (
    df["gsc_impressions"]
    + np.where(df["gsc_avg_position"] > 20, 100, 0)
    + np.where(df["ga4_engaged_sessions"] == 0, 100, 0)
)

df["reason_code"] = "visible_low_engagement"
df["action"] = "REVIEW_CONTENT"

df = df.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print(f"Rows written: {len(df):,}")
print(f"Saved: {output_path}")

df.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows written: 638,608
Saved: work/outputs/baseline_action_score.csv


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,score,reason_code,action
625528,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,0.083350,0,40184,visible_low_engagement,REVIEW_CONTENT
516895,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,2.197507,15,39305,visible_low_engagement,REVIEW_CONTENT
10330,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2.764916,0,39103,visible_low_engagement,REVIEW_CONTENT
590119,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,2.195988,7,38436,visible_low_engagement,REVIEW_CONTENT
10359,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,8.613948,0,37468,visible_low_engagement,REVIEW_CONTENT
572986,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,2.188397,3,35404,visible_low_engagement,REVIEW_CONTENT
554137,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,2.181348,4,34817,visible_low_engagement,REVIEW_CONTENT
632303,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,2.242501,10,34606,visible_low_engagement,REVIEW_CONTENT
465687,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,2.309046,12,33571,visible_low_engagement,REVIEW_CONTENT
569202,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0.181500,0,33483,visible_low_engagement,REVIEW_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

The highest-ranked pages were consistently visible in Google Search while showing little or no engagement. These pages were prioritized for manual content review.

Confidence: Medium.

This baseline could be wrong if the pages intentionally receive little engagement, if GA4 tracking is incomplete, or if search visibility is temporary.

In [15]:
df.head(20)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,score,reason_code,action
625528,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,0.083350,0,40184,visible_low_engagement,REVIEW_CONTENT
516895,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,2.197507,15,39305,visible_low_engagement,REVIEW_CONTENT
10330,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2.764916,0,39103,visible_low_engagement,REVIEW_CONTENT
590119,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,2.195988,7,38436,visible_low_engagement,REVIEW_CONTENT
10359,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,8.613948,0,37468,visible_low_engagement,REVIEW_CONTENT
572986,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,2.188397,3,35404,visible_low_engagement,REVIEW_CONTENT
554137,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,2.181348,4,34817,visible_low_engagement,REVIEW_CONTENT
632303,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,2.242501,10,34606,visible_low_engagement,REVIEW_CONTENT
465687,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,2.309046,12,33571,visible_low_engagement,REVIEW_CONTENT
569202,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0.181500,0,33483,visible_low_engagement,REVIEW_CONTENT


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

Some lower-ranked pages may not require action because they naturally attract low engagement or belong to seasonal content.

Leakage check:
- No future information was used.
- No product flags or outcome labels were used.
- Only March 2026 observations were used for scoring.

In [16]:
print("Top 20")
display(df.head(20))

print("\nBottom 20")
display(df.tail(20))


Top 20


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,score,reason_code,action
625528,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,0.083350,0,40184,visible_low_engagement,REVIEW_CONTENT
516895,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,2.197507,15,39305,visible_low_engagement,REVIEW_CONTENT
10330,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2.764916,0,39103,visible_low_engagement,REVIEW_CONTENT
590119,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,2.195988,7,38436,visible_low_engagement,REVIEW_CONTENT
10359,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,8.613948,0,37468,visible_low_engagement,REVIEW_CONTENT
572986,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,2.188397,3,35404,visible_low_engagement,REVIEW_CONTENT
554137,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,2.181348,4,34817,visible_low_engagement,REVIEW_CONTENT
632303,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,2.242501,10,34606,visible_low_engagement,REVIEW_CONTENT
465687,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,2.309046,12,33571,visible_low_engagement,REVIEW_CONTENT
569202,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0.181500,0,33483,visible_low_engagement,REVIEW_CONTENT



Bottom 20


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,score,reason_code,action
131809,2026-03-07,client_23a62021009f63c4,content_1b640716b0e300f4,100,9.87,1,100,visible_low_engagement,REVIEW_CONTENT
317338,2026-03-16,client_23a62021009f63c4,content_47cf1d6cb12a8d18,100,4.76,1,100,visible_low_engagement,REVIEW_CONTENT
530596,2026-03-27,client_73cda7b4e4f265ea,content_6ff378b3467bb271,100,12.65,1,100,visible_low_engagement,REVIEW_CONTENT
537117,2026-03-28,client_23a62021009f63c4,content_f875a0fa911ba3aa,100,1.74,1,100,visible_low_engagement,REVIEW_CONTENT
553214,2026-03-25,client_20259bd6705d81d4,content_34de33a2ace3b621,100,5.81,1,100,visible_low_engagement,REVIEW_CONTENT
632928,2026-03-31,client_0fa64a184f18a4a0,content_6f477b9c04d171d9,100,7.77,1,100,visible_low_engagement,REVIEW_CONTENT
304502,2026-03-17,client_23a62021009f63c4,content_553d79dfb06c9562,100,16.42,1,100,visible_low_engagement,REVIEW_CONTENT
237328,2026-03-13,client_e547b89c05043229,content_b67d4660733b6a13,100,3.76,1,100,visible_low_engagement,REVIEW_CONTENT
104211,2026-03-02,client_23a62021009f63c4,content_43aa81dd5129441e,100,8.65,1,100,visible_low_engagement,REVIEW_CONTENT
517457,2026-03-29,client_e547b89c05043229,content_3795ae107e307c7b,100,4.75,1,100,visible_low_engagement,REVIEW_CONTENT


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.